<a href="https://colab.research.google.com/github/p-tech/ntu-growth-hacking/blob/main/clustering/Notebook_7_Oddbox_Customer_Segmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 7 · Which Oddbox customers should receive the same message?
**Week 1 · Session 4 · MSc Growth Hacking**

The founder wants to encourage another order. Some customers tried a single box, some regularly order vegetables, and others add fruit. Could purchase patterns help us decide who should receive which message?

In pairs, work down the notebook one cell at a time. Run each code cell with **Shift + Enter**. Stop at each decision before continuing. By the end, you will have a customer file with cluster assignments and a proposal for one targeted experiment.

**Starting question** — Choose two customers who placed the same number of orders. What differences between their baskets might justify sending them different messages?

## 1. Upload the customer CSV
Run the cell and select **Oddbox-1000-customers.csv** from your computer. Each row records one synthetic customer over 12 months. A zero means the customer never bought that product. A fruit booster accompanies a veg box, so it does not count as another order.

In [13]:
from google.colab import files
from IPython.display import display
import io
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

filename = "Oddbox-1000-customers.csv"
# Use the raw GitHub URL for the CSV file, or fallback to the locally uploaded file
data_url = "https://raw.githubusercontent.com/p-tech/ntu-growth-hacking/main/clustering/Oddbox-1000-customers.csv"

try:
    customers = pd.read_csv(data_url)
except Exception as e:
    print("Could not load from URL, attempting to load local file...")
    customers = pd.read_csv(filename)

customers = customers.replace(r'^\s*$', pd.NA, regex=True)
before = len(customers)
customers = customers.dropna(how='all').drop_duplicates().reset_index(drop=True)
print(f'Removed {before-len(customers)} blank rows or exact repeated records.')
print(f'{filename} contains {len(customers):,} rows.')
display(customers.head())

Removed 0 blank rows or exact repeated records.
Oddbox-1000-customers.csv contains 1,000 rows.


,customer_id,fruit_and_veg_boxes,veg_boxes,fruit_boosters,orders_12m,household_size,household_wealth,gender
0,1,1,0,0,1,1,Medium,Female
1,2,1,0,0,1,2,High,Female
2,3,9,0,0,9,5,Medium,Male
3,4,1,2,0,3,1,Low,Male
4,5,1,0,0,1,3,Low,Female


## 2. Check what each row means
Before grouping customers, check whether an ID appears twice, values are missing, or an order total is impossible. Such errors can move a cluster centre. We keep unusual but valid purchases, including one-off trials.

In [ ]:
products = ['fruit_and_veg_boxes', 'veg_boxes', 'fruit_boosters']
profile_fields = ['household_size', 'household_wealth', 'gender']
required = ['customer_id', 'orders_12m'] + products + profile_fields
missing = set(required) - set(customers.columns)
if missing:
    raise ValueError(f'Missing columns: {sorted(missing)}')
counts = customers[products + ['orders_12m', 'household_size']]
checks = {
    'One row per customer ID': customers.customer_id.is_unique,
    'All required values present': customers[required].notna().all().all(),
    'Whole-number counts': counts.mod(1).eq(0).all().all(),
    'Product quantities between 0 and 12': customers[products].ge(0).all().all() and customers[products].le(12).all().all(),
    'Annual orders between 1 and 12': customers.orders_12m.between(1, 12).all(),
    'Main boxes sum to orders': customers[products[:2]].sum(axis=1).eq(customers.orders_12m).all(),
    'Boosters do not exceed veg boxes': customers.fruit_boosters.le(customers.veg_boxes).all(),
    'Household size between 1 and 5': customers.household_size.between(1, 5).all()
}
display(pd.Series(checks, name='Passed'))
if not all(checks.values()):
    raise ValueError('Review the failed checks before continuing.')
display(customers[products].describe().round(1))
print('One-order customers:', customers.orders_12m.eq(1).sum())

## 3. Which purchases should count towards similarity?
A **feature** is a column the model uses when comparing customers. Suppose two customers bought six veg boxes each. One added fruit five times and the other never added fruit. Should they be close together if you are choosing whom to send a fruit promotion?

**First run:** Leave all three boxes ticked in the Colab form below. Run this cell and continue down the notebook. If you see the code view, the same switches appear as True or False.

**Second run:** Record your first result, then untick **include_fruit_boosters** (or change True to False). Run this cell and every cell below it again, keeping the same k. Predict which purchase patterns the model will now struggle to distinguish. Compare the customer counts and purchase averages for the two runs.

We standardise each selected feature so a one-standard-deviation change receives the same starting weight. ID identifies the customer. Annual orders already sum the two main boxes. Household size and wealth enter separately in the final investigation so you can see their effects.

In [ ]:
#@title Choose the purchases to include
include_fruit_and_veg_boxes = True #@param {type:"boolean"}
include_veg_boxes = True #@param {type:"boolean"}
include_fruit_boosters = True #@param {type:"boolean"}

switches = [include_fruit_and_veg_boxes, include_veg_boxes, include_fruit_boosters]
features = [name for name, selected in zip(products, switches) if selected]
if len(features) < 2:
    raise ValueError('Select at least two products for this comparison and rerun the cell.')
print('Distance will use:', ', '.join(features))
scaler = StandardScaler()
X = scaler.fit_transform(customers[features])
display(pd.DataFrame(X, columns=features).head().round(2))

## 4. Try k-means with several values of k
Here **k** is the number of groups you ask the method to find. K-means starts with k centres, assigns each customer to the nearest centre, then moves each centre to the average of its members. It repeats until assignments or centres settle. Trying several starting points reduces the influence of an unlucky start.

**Left chart · Inertia and the elbow method**

Imagine placing a marker at the centre of each group on a customer purchase map. Inertia measures how tightly customers cluster around those markers. The code takes each customer’s distance from their group centre, squares it and adds up the results. A lower value means customers are closer to their centres overall.

Read the line from left to right. Each step adds one group. A steep drop means the extra group brought customers much closer to their centres. A small drop means it added little. The elbow method means looking for the bend where a steep fall starts to level off. For example, if inertia falls sharply up to three groups and then falls only slightly at four and five, three groups is a candidate. Some charts have no clear bend.

**Right chart · Silhouette score**
Silhouette asks a different question: does each customer buy more like the people in their own group or like those in the nearest other group? A score near +1 means their own group is a clear fit. Near 0 means either group could fit. A negative score means their buying habits are closer, on average, to the other group.

Each point on the right chart is the average silhouette score across all customers for that number of groups. Look for a high point: it suggests the groups are more clearly separated on the purchase measures you selected.

**Your decision**.

Which number of groups does the elbow suggest? Which has the highest average silhouette? If the answers differ, try both in the next section. Look at the average orders in each group. Would the extra group help Oddbox choose a different offer for a particular set of customers?

In [ ]:
models = {}
scores = []
for k in range(2, 8):
    model = KMeans(n_clusters=k, n_init=25, random_state=42).fit(X)
    models[k] = model
    scores.append([k, model.inertia_, silhouette_score(X, model.labels_)])
results = pd.DataFrame(scores, columns=['k', 'inertia', 'silhouette']).set_index('k')
display(results.round(3))
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
for ax, measure in zip(axes, results.columns):
    results[measure].plot(ax=ax, marker='o', title=measure.capitalize(), xlabel='Number of clusters')
best_k = results.silhouette.idxmax()
axes[1].scatter(best_k, results.loc[best_k, 'silhouette'], color='darkorange', zorder=3,
                label=f'Highest mean (k={best_k})')
axes[1].legend()
plt.tight_layout()
plt.show()

## 5. Choose k and read the groups in boxes per year
**Decision** — Pick a value from 2 to 7. Four is supplied as a starting point. Record why you chose it. Then try a neighbouring value and ask whether the extra group would actually receive a different message.

Each cluster number is just a label. The table below returns to the original units. For example, a mean of 4.4 boosters describes a group average. Each customer still purchased a whole number.

In [ ]:
chosen_k = 4  # Change this and rerun from here to compare solutions.
if chosen_k not in models:
    raise ValueError('Choose a value from 2 to 7.')
assigned = customers.copy()
assigned['cluster'] = models[chosen_k].labels_ + 1
groups = assigned.groupby('cluster')
summary = groups[products + ['orders_12m']].mean()
summary.insert(0, 'customers', groups.size())
summary['one_order_%'] = assigned.orders_12m.eq(1).groupby(assigned.cluster).mean() * 100
display(summary.round(1))
summary[products].plot.bar(figsize=(9, 4), rot=0, ylabel='Mean units per customer over 12 months')
plt.legend(['Fruit & veg box', 'Veg box', 'Fruit booster'])
plt.tight_layout()
plt.show()

### Does the overall score hide a weak cluster?
**How well do customers fit their assigned cluster?**

The earlier chart helped you choose how many clusters to use. This plot takes your chosen number of clusters and shows how well each customer fits their group.

Each coloured band represents one cluster. Think of it as a stack of thin horizontal bars, one per customer. A taller band means that cluster contains more customers. To judge the fit, look at how far the bars extend across the horizontal axis.
Silhouette score	What it means for that customer

Near +1	Their buying habits are much more similar to customers in their own cluster than to those in the nearest other cluster.

Near 0	Their buying habits fit two clusters about equally well.

Below 0	Their buying habits are more similar, on average, to customers in another cluster.


**What does a good fit look like?**

Look for bands where most bars extend clearly to the right of zero and few extend to the left. The dashed line marks the average score across all customers. A customer can fall below that average and still fit their cluster well.

**Discuss:**

Which cluster has the lowest mean?

Which has the most customers below zero as a percentage of its members?

Inspect the customer with the lowest score and their purchase counts. Would you confidently send them the same message as the rest of their group?

Which cluster has the clearest fit?

Which has the most customers near or below zero?

Would you feel equally confident sending one marketing message to everyone in each of those clusters?

In [ ]:
import numpy as np
from sklearn.metrics import silhouette_samples

assigned['silhouette'] = silhouette_samples(X, assigned.cluster)
quality = assigned.groupby('cluster').silhouette.agg(
    customers='size', mean_score='mean', lowest_score='min',
    below_zero_pct=lambda scores: 100 * scores.lt(0).mean()
)
display(quality.round(3))
fig, ax = plt.subplots(figsize=(9, 5))
ticks = []
offset = 0
for cluster, group in assigned.groupby('cluster'):
    scores = np.sort(group.silhouette)
    positions = np.arange(len(scores)) + offset
    ax.barh(positions, scores, height=1, linewidth=0)
    ticks.append(positions.mean())
    offset += len(scores) + 15
ax.set_yticks(ticks, labels=quality.index)
ax.axvline(0, color='black', linewidth=1)
ax.axvline(assigned.silhouette.mean(), color='darkred', linestyle='--', label='Overall mean')
ax.set(xlim=(-1, 1), xlabel='Customer silhouette score', ylabel='Cluster',
       title=f'How well customers fit their assigned cluster (k={chosen_k})')
ax.legend()
plt.tight_layout()
plt.show()
display(assigned.nsmallest(5, 'silhouette')[['customer_id', 'cluster'] + products + ['silhouette']])

### Would we find similar groups if we used a different 800 customers?
A well-separated solution can still be sensitive to the customers used to fit it. For five trials, the next cell fits the same model to a different 80% sample and predicts groups for all customers. It compares each result with the full-data model using the **adjusted Rand index**. A score of 1 means identical groupings even if their numbers have changed. Around zero means agreement similar to chance. Keep the original scaling fixed so the check focuses on the sample used to fit the centres.

**Interpretation:** A high score supports repeatability on these data. To find out whether a message increases orders, you still need the campaign experiment in step 7.

In [ ]:
from sklearn.metrics import adjusted_rand_score

stability = []
for seed in range(5):
    sample = np.random.default_rng(seed).choice(len(X), size=int(0.8 * len(X)), replace=False)
    refit = KMeans(n_clusters=chosen_k, n_init=25, random_state=42).fit(X[sample])
    stability.append(adjusted_rand_score(assigned.cluster, refit.predict(X)))
display(pd.DataFrame({'Trial': range(1, 6), 'Agreement with full model': stability}).round(3))

## 6. Check who a cluster label could overlook
Look at the spread within each cluster. Compare its one-order percentage with the description you had in mind. Pick one customer whose purchases challenge your description.

The demographic tables show percentages **within each cluster** alongside the percentage for **all customers**. Is a category common in the cluster simply because it is common across the dataset? These synthetic characteristics can suggest questions for research. Household size alone does not tell us whether children live there.

In [ ]:
focus_cluster = 1  # Change to the group you are considering.
focus = assigned.loc[assigned.cluster.eq(focus_cluster)].sort_values('orders_12m')
display(focus[products + ['orders_12m']].agg(['min', 'median', 'max']))
display(pd.concat([focus.head(3), focus.tail(3)]).drop_duplicates())
for field in profile_fields:
    shares = pd.crosstab(assigned.cluster, assigned[field], normalize='index') * 100
    shares.loc['All customers'] = assigned[field].value_counts(normalize=True) * 100
    print(field.replace('_', ' ').title(), '(% within each row)')
    display(shares.round(1))

## 7. Turn one group into a testable targeting decision
Double-click the response cell below to edit it. Name the group using its observed purchases. Use your earlier review or market research to suggest *why* a message might work. A review from a different dataset can suggest a motive, but you cannot attribute it to a particular customer here.

For example, a “second box” email only makes sense for households with exactly one order. If your chosen cluster mixes trial and repeat buyers, add an eligibility rule. For a trial offer, remember the first box was £8 and the following box is £16.

### Our recommendation
- **Model choice:** We used ___ features and k = ___ because ___. With k = ___ the main difference was ___.
- **Audience:** We call cluster ___ “___”. It contains ___ customers. Two figures that justify the name are ___.
- **Eligibility:** Within that cluster, we would contact customers who ___. Someone we could miss outside it is ___.
- **Message:** Our proposed subject line is “___”. Our reason for expecting it to work is ___, based on ___.
- **Test:** Randomly split eligible customers between our message and ___. Compare the percentage in each group who ___ within ___ days.
- **What would change our minds:** We would revise the audience or message if ___.

## 8. What changes when we include the household?
So far, household size and wealth have described the groups **after** fitting. Now they will help decide who belongs together. Predict what will happen to two customers who buy exactly the same boxes but have different household sizes.

Use your selected products as the baseline. Keep the same customers, k, scaling method and model settings. Start by comparing purchases alone, purchases plus size, purchases plus wealth, and purchases plus both. The single-feature runs let you check each feature separately.

**How should we represent wealth or income?** The CSV records **household wealth band**. Wealth describes assets less debts. Income measures money received over a period. For this exercise use the information supplied: Low = 1, Medium = 2, High = 3. This preserves the ordering and assumes two equal steps. Those steps have no pound value. Standardising gives this three-level feature as much starting weight as each other feature, so including it is a substantive decision.

If the research question concerns spending power and you have measured annual household income in pounds, a consistent gross or net income figure preserves differences within bands. Check extreme values and standardise before clustering. If you only know the bands, assigning invented salaries would imply precision the data cannot support. Household size measures people living together. Their ages and relationships are unknown.

Then inspect the gender-only addition and the run using all household fields. Gender is a category without a ranking. The code gives each recorded category its own indicator. A change of category contributes one unit of squared distance in this exercise. Discuss whether using gender to form groups adds value for this campaign. You can keep it as a descriptive field only.

In [ ]:
from scipy.optimize import linear_sum_assignment

comparison = customers.copy()
comparison['wealth_rank'] = comparison.household_wealth.str.strip().str.title().map({'Low': 1, 'Medium': 2, 'High': 3})
if comparison.wealth_rank.isna().any():
    raise ValueError('Check that all wealth bands are Low, Medium or High.')
gender_flags = pd.get_dummies(comparison.gender.astype(str).str.strip().str.casefold(), prefix='gender', dtype=float)
comparison = comparison.join(gender_flags)
gender_columns = list(gender_flags.columns)
scenarios = {
    'Purchases': features,
    'Plus size': features + ['household_size'],
    'Plus wealth': features + ['wealth_rank'],
    'Plus both': features + ['household_size', 'wealth_rank'],
    'Plus gender': features + gender_columns,
    'All household fields': features + ['household_size','wealth_rank'] + gender_columns
}
baseline = assigned.cluster.to_numpy()
labels_by_run = {}
metrics = []
for name, columns in scenarios.items():
    numeric = [col for col in columns if col not in gender_columns]
    values = StandardScaler().fit_transform(comparison[numeric])
    if any(col in gender_columns for col in columns):
        values = np.column_stack([values, comparison[gender_columns].to_numpy()/np.sqrt(2)])
    raw = KMeans(n_clusters=chosen_k, n_init=25, random_state=42).fit_predict(values) + 1
    overlap = pd.crosstab(baseline, raw)
    rows, cols = linear_sum_assignment(-overlap.to_numpy())
    mapping = dict(zip(overlap.columns[cols], overlap.index[rows]))
    aligned = pd.Series(raw).map(mapping).to_numpy()
    labels_by_run[name] = aligned
    metrics.append([name, (aligned != baseline).sum(), adjusted_rand_score(baseline, raw),
                    silhouette_score(values, raw)])
display(pd.DataFrame(metrics, columns=['Run', 'Customers changing group', 'Agreement with purchases',
                                      'Silhouette in own feature space']).set_index('Run').round(3))

**Read the comparison.** Cluster numbers are arbitrary. The code pairs each new group with one original group to maximise the number of customers who keep the same label. The movement count shows how many still change after that matching. Agreement of 1 means identical groupings. Each silhouette score measures separation using that run’s own definition of similarity, so selecting the highest score here would answer a different question from selecting k with fixed features.

**Investigate one change.** Choose a run below. Read across a row to see where the original customers went. The two profile tables show whether the revised groups differ in purchases, household size or wealth. Compare those changes with your prediction.

In [ ]:
#@title Which change would you like to inspect?
inspect_run = "All household fields" #@param ["Plus size", "Plus wealth", "Plus both", "Plus gender", "All household fields"]
inspection = comparison.assign(original=baseline, revised=labels_by_run[inspect_run])
display(pd.crosstab(inspection.original, inspection.revised, margins=True))
for column in ['original', 'revised']:
    profile = inspection.groupby(column)[products + ['household_size']].mean().round(2)
    profile['customers'] = inspection.groupby(column).size()
    wealth_mix = pd.crosstab(inspection[column], inspection.household_wealth, normalize='index')
    display(profile.join((100 * wealth_mix).round(1).add_suffix(' wealth (%)')))
moved = inspection.loc[inspection.original.ne(inspection.revised)]
display(moved[['customer_id'] + products + ['household_size', 'household_wealth', 'gender', 'original', 'revised']].head(8))

**Decision for the founder.** Pick one customer who moved. What purchase evidence would justify changing their message? Does knowing their household size or wealth band strengthen that case? If nobody moved, describe the evidence that the original grouping held. Finish with one feature you would keep for grouping, one you would use only to describe customers, and why.

## 9. Did the new groups separate purchasing behaviour more clearly?
Use the run you selected above. Both charts place each customer in exactly the same position. The axes summarise the selected purchase quantities using two principal components (directions capturing the most variation). Each colour shows a cluster, matched across charts. A customer who changes colour has changed group. Small, identical offsets on both charts help reveal customers with identical purchases. The model and scores use the original standardised values without these offsets.

The table tests every grouping against the **same purchase measures**, then against the **same purchase and household measures**. Compare scores down one column at a time. Read the full-dimensional scores alongside the two-dimensional charts.

In [ ]:
from sklearn.decomposition import PCA

purchase_values = StandardScaler().fit_transform(comparison[features])
full_values = StandardScaler().fit_transform(comparison[scenarios['Plus both']])
common_scores = pd.DataFrame({
    'Purchase separation': {name: silhouette_score(purchase_values, labels)
                            for name, labels in labels_by_run.items()},
    'Purchase and household separation': {name: silhouette_score(full_values, labels)
                                          for name, labels in labels_by_run.items()}
})
display(common_scores.round(3))

projection = PCA(n_components=2).fit(purchase_values)
points = projection.transform(purchase_values)
points += np.random.default_rng(42).normal(0, 0.04, points.shape)
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)
for ax, run in zip(axes, ['Purchases', inspect_run]):
    for cluster in sorted(np.unique(baseline)):
        members = labels_by_run[run] == cluster
        ax.scatter(*points[members].T, s=12, alpha=0.5, label=str(cluster))
    ax.set(title=run, xlabel='Purchase component 1')
axes[0].set_ylabel('Purchase component 2')
axes[1].legend(title='Cluster', fontsize=8)
fig.suptitle(f'Same customers and axes in both panels ({projection.explained_variance_ratio_.sum():.0%} of purchase variation shown)')
plt.tight_layout()
plt.show()

## 10. Choose the grouping to use and download the customers
For your proposed campaign, which definition of similarity matters? Give one score from the relevant column, one observed purchase pattern and one reason your message suits the chosen group. Revisit your earlier recommendation if your judgement has changed.

Choose the model to export using the dropdown below. The CSV includes the original fields, the original purchase cluster, the final cluster, a flag for customers who moved, and the settings used. You can filter the CSV by final cluster and apply your campaign eligibility rule.

In [ ]:
#@title Download your chosen grouping
export_model = "Purchases" #@param ["Purchases", "Plus size", "Plus wealth", "Plus both", "Plus gender", "All household fields"]
final = customers.copy()
final['purchase_cluster'] = baseline
final['final_cluster'] = labels_by_run[export_model]
final['changed_group'] = final.final_cluster.ne(final.purchase_cluster)
final['model'] = export_model
final['k'] = chosen_k
final['model_features'] = ', '.join(scenarios[export_model])
output_name = f"Oddbox-k{chosen_k}-{export_model.replace(' ', '-')}-{'-'.join(features)}.csv"
print(f'Exporting {len(final):,} customers using {export_model}.')
display(final.head())
final.to_csv(output_name, index=False)
files.download(output_name)